# Requirement 25: Demonstrable enforcement

> Prove these controls through a working implementation and negative tests.

**Covered by:** Caller

**How it is fulfilled:** This notebook runs the key negative tests in one pass and prints a pass/fail table. Each test tries to break a control and expects it to hold.

In [1]:
import sys
sys.path.insert(0, "..")

In [2]:
from helpers import *
import requests
results = {}
def test(name, fn):
    try:
        results[name] = bool(fn())
    except Exception as err:
        results[name] = False
        print("error in", name, "->", type(err).__name__, str(err)[:100])

In [3]:
# Administrator tries to read sandbox contents
def admin_denied():
    url = f"{DATA_GROUP_URL}/sandboxes?{DATA_API}"
    return requests.get(url, headers=token("admin", "data"), timeout=30).status_code == 403
test("1  admin cannot list sandboxes on the data plane", admin_denied)

# Administrator tries to grant itself Data Owner
def self_grant_denied():
    import uuid
    defs = requests.get(f"{ARM}{GROUP_SCOPE}/providers/Microsoft.Authorization/roleDefinitions?api-version=2022-04-01&$filter=roleName eq 'Container Apps SandboxGroup Data Owner'",
                        headers=token("orchestrator", "arm"), timeout=30).json()
    body = {"properties": {"roleDefinitionId": defs["value"][0]["id"], "principalId": ADMIN_OID, "principalType": "ServicePrincipal"}}
    r = requests.put(f"{ARM}{GROUP_SCOPE}/providers/Microsoft.Authorization/roleAssignments/{uuid.uuid4()}?api-version=2022-04-01",
                     headers=token("admin", "arm"), json=body, timeout=30)
    return r.status_code in (403,)
test("1  admin cannot grant itself Data Owner", self_grant_denied)

In [4]:
sandbox, session_id = new_session()
other, other_id = new_session()
key = install_session_key(sandbox, session_id)
install_session_key(other, other_id)
test("3  tampered configuration is detected", lambda: not all(verify_session(sandbox, expected_disk_id="bad").values()))
test("10 replayed request is rejected", lambda: (send(sandbox, signed_request(key, session_id, 1, [1])), "replay" in send(sandbox, signed_request(key, session_id, 1, [1])))[1])
test("10 request signed with the wrong key is rejected", lambda: "REJECTED" in send(sandbox, signed_request(os.urandom(32), session_id, 5, [1])))
test("13 sessions are isolated", lambda: "No such file" in (sandbox.write_file("/tmp/a.txt", "x"), other.exec("cat /tmp/a.txt 2>&1").stdout)[1])
test("19 runaway code is stopped", lambda: sandbox.exec("timeout 3 python -c 'while True: pass'").exit_code == 124)
test("20 internet is blocked", lambda: sandbox.exec("curl -s -o /dev/null -m 8 -w '%{http_code}' https://example.com").stdout.strip() != "200")
test("14 suspend is disabled", lambda: not sandbox.get().lifecycle.auto_suspend.enabled)

In [5]:
cleanup(sandbox, other)
test("15 sessions are gone after cleanup", lambda: not list(orchestrator().list_sandboxes(labels={"session": session_id})))
print()
for name, ok in results.items():
    print("PASS" if ok else "FAIL", name)
check("all negative tests held", all(results.values()))


PASS 1  admin cannot list sandboxes on the data plane
PASS 1  admin cannot grant itself Data Owner
PASS 3  tampered configuration is detected
PASS 10 replayed request is rejected
PASS 10 request signed with the wrong key is rejected
PASS 13 sessions are isolated
PASS 19 runaway code is stopped
PASS 20 internet is blocked
PASS 14 suspend is disabled
PASS 15 sessions are gone after cleanup
PASS  all negative tests held


**Caveat:** Restart and interruption tests (orchestrator crash mid-session, network drop) are not yet covered, and should be added before Contoso relies on requirement 14 in production.